In [1]:
import pandas as pd
import numpy as np

# ── LOAD ──────────────────────────────────────────────────────────────────────
nodes = pd.read_csv("../mc1_csv/mc1_nodes.csv")
edges = pd.read_csv("../mc1_csv/mc1_edges.csv")

songs     = nodes[nodes["Node Type"] == "Song"].copy()
albums    = nodes[nodes["Node Type"] == "Album"].copy()
persons   = nodes[nodes["Node Type"] == "Person"].copy()
all_works = pd.concat([songs, albums], ignore_index=True)

all_works["release_date"]   = pd.to_numeric(all_works["release_date"],   errors="coerce")
all_works["notoriety_date"] = pd.to_numeric(all_works["notoriety_date"], errors="coerce")

performer_of = edges[edges["Edge Type"] == "PerformerOf"].rename(
    columns={"source": "artist_id", "target": "work_id"}
)[["artist_id", "work_id"]]

influence_types = ["InStyleOf", "InterpolatesFrom", "CoverOf",
                   "LyricalReferenceTo", "DirectlySamples"]
all_influence = edges[edges["Edge Type"].isin(influence_types)].rename(
    columns={"source": "source_work_id", "target": "target_work_id",
             "Edge Type": "influence_type"}
)[["source_work_id", "target_work_id", "influence_type"]]

# ── THREE ARTISTS ─────────────────────────────────────────────────────────────
THREE = {
    17255: "Sailor Shift",
    5038:  "Min He",
    1716:  "Kimberly Snyder",
}
THREE_IDS = list(THREE.keys())

# ── STEP 1: ARTIST → WORKS WITH ATTRIBUTES ────────────────────────────────────
artist_works = performer_of[performer_of["artist_id"].isin(THREE_IDS)].merge(
    all_works[["id", "genre", "release_date", "notable", "notoriety_date"]],
    left_on="work_id", right_on="id", how="left"
)
artist_works["release_date"] = pd.to_numeric(artist_works["release_date"], errors="coerce")
artist_works["notable"]      = artist_works["notable"].astype(bool)

# ── STEP 2: NOTABLE WORKS PER YEAR ───────────────────────────────────────────
notable_yearly = (
    artist_works[artist_works["notable"] == True]
    .groupby(["artist_id", "release_date"])
    .size()
    .reset_index(name="new_notable_works")
    .rename(columns={"release_date": "year"})
)

# ── STEP 3: COLLABS PER YEAR ──────────────────────────────────────────────────
# Find shared songs (multiple performers) and build (artist, collaborator, year) table
song_performers = performer_of.groupby("work_id")["artist_id"].apply(list).reset_index()
shared_songs    = song_performers[song_performers["artist_id"].apply(len) > 1]

collab_pairs = []
for _, row in shared_songs.iterrows():
    perfs   = row["artist_id"]
    work_id = row["work_id"]
    for i in range(len(perfs)):
        for j in range(i + 1, len(perfs)):
            collab_pairs.append({
                "artist_a": perfs[i],
                "artist_b": perfs[j],
                "work_id":  work_id
            })

collab_df = pd.DataFrame(collab_pairs)
collab_df = collab_df.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "work_id"}),
    on="work_id", how="left"
)
collab_df["release_date"] = pd.to_numeric(collab_df["release_date"], errors="coerce")

# Long format: one row per (artist, collaborator, year)
collab_long = pd.concat([
    collab_df.rename(columns={"artist_a": "artist_id", "artist_b": "collaborator"}),
    collab_df.rename(columns={"artist_b": "artist_id", "artist_a": "collaborator"})
])[["artist_id", "collaborator", "work_id", "release_date"]]

# Filter to our three artists
collab_three = collab_long[collab_long["artist_id"].isin(THREE_IDS)]

# Unique collaborators per year
collab_yearly = (
    collab_three.groupby(["artist_id", "release_date"])["collaborator"]
    .nunique()
    .reset_index(name="new_collabs")
    .rename(columns={"release_date": "year"})
)

# ── STEP 4: OUTBOUND INFLUENCE PER YEAR ──────────────────────────────────────
# Map work_id → artist_id for our three artists
work_to_artist = (
    artist_works[["artist_id", "work_id"]]
    .set_index("work_id")["artist_id"]
    .to_dict()
)

outbound = all_influence.copy()
outbound["artist_id"] = outbound["source_work_id"].map(work_to_artist)
outbound = outbound.dropna(subset=["artist_id"])
outbound["artist_id"] = outbound["artist_id"].astype(int)

# Attach source year from all_works
outbound = outbound.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "source_work_id"}),
    on="source_work_id", how="left"
)
outbound["release_date"] = pd.to_numeric(outbound["release_date"], errors="coerce")

outbound_yearly = (
    outbound[outbound["artist_id"].isin(THREE_IDS)]
    .groupby(["artist_id", "release_date"])
    .size()
    .reset_index(name="outbound_influence")
    .rename(columns={"release_date": "year"})
)

# ── STEP 5: INBOUND INFLUENCE PER YEAR ───────────────────────────────────────
# Map target_work_id → artist_id for our three artists
inbound = all_influence.copy()
inbound["artist_id"] = inbound["target_work_id"].map(work_to_artist)
inbound = inbound.dropna(subset=["artist_id"])
inbound["artist_id"] = inbound["artist_id"].astype(int)

# Attach the CITING work's year (source_work_id year = when the citation was made)
inbound = inbound.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "source_work_id"}),
    on="source_work_id", how="left"
)
inbound["release_date"] = pd.to_numeric(inbound["release_date"], errors="coerce")

inbound_yearly = (
    inbound[inbound["artist_id"].isin(THREE_IDS)]
    .groupby(["artist_id", "release_date"])
    .size()
    .reset_index(name="inbound_influence")
    .rename(columns={"release_date": "year"})
)

# ── STEP 6: COMBINE INTO ONE WIDE TIMELINE TABLE ──────────────────────────────
# Get full year range per artist (from debut to latest work)
year_ranges = []
for aid in THREE_IDS:
    works = artist_works[artist_works["artist_id"] == aid]
    if len(works) == 0:
        continue
    debut  = int(works["release_date"].min())
    latest = int(works["release_date"].max())
    for yr in range(debut, latest + 1):
        year_ranges.append({"artist_id": aid, "year": yr})

year_grid = pd.DataFrame(year_ranges)

# Merge all metrics onto the year grid
timeline = year_grid.copy()
timeline = timeline.merge(notable_yearly,  on=["artist_id", "year"], how="left")
timeline = timeline.merge(collab_yearly,   on=["artist_id", "year"], how="left")
timeline = timeline.merge(outbound_yearly, on=["artist_id", "year"], how="left")
timeline = timeline.merge(inbound_yearly,  on=["artist_id", "year"], how="left")

# Fill nulls with 0 (no activity that year = 0)
for col in ["new_notable_works", "new_collabs", "outbound_influence", "inbound_influence"]:
    timeline[col] = timeline[col].fillna(0).astype(int)

# Add cumulative columns
timeline = timeline.sort_values(["artist_id", "year"])
timeline["cumulative_notable_works"] = timeline.groupby("artist_id")["new_notable_works"].cumsum()
timeline["cumulative_collabs"]        = timeline.groupby("artist_id")["new_collabs"].cumsum()
timeline["cumulative_outbound"]       = timeline.groupby("artist_id")["outbound_influence"].cumsum()
timeline["cumulative_inbound"]        = timeline.groupby("artist_id")["inbound_influence"].cumsum()

# Attach display name
name_map = {aid: THREE[aid] for aid in THREE_IDS}
timeline["display_name"] = timeline["artist_id"].map(name_map)

# Attach debut year and notoriety year for reference lines in Tableau
for aid in THREE_IDS:
    works = artist_works[artist_works["artist_id"] == aid]
    debut     = int(works["release_date"].min())
    notoriety = works["notoriety_date"].min()
    notoriety = int(notoriety) if pd.notna(notoriety) else None
    timeline.loc[timeline["artist_id"] == aid, "debut_year"]     = debut
    timeline.loc[timeline["artist_id"] == aid, "notoriety_year"] = notoriety

timeline["debut_year"]     = timeline["debut_year"].astype("Int64")
timeline["notoriety_year"] = timeline["notoriety_year"].astype("Int64")

# ── STEP 7: GENRE CONTRIBUTION TABLE ─────────────────────────────────────────
# One row per artist per genre per year — for the genre chart
genre_rows = artist_works[artist_works["artist_id"].isin(THREE_IDS)].copy()
genre_rows = genre_rows.dropna(subset=["genre", "release_date"])
genre_rows["year"] = genre_rows["release_date"].astype(int)
genre_rows["display_name"] = genre_rows["artist_id"].map(name_map)

genre_yearly_raw = (
    genre_rows.groupby(["artist_id", "display_name", "year", "genre"])
    .size()
    .reset_index(name="works_in_genre")
)

# Build full artist × genre × year grid so every year has a row for every genre
# (carries forward cumulative counts even in years with no new works)
genre_grid_rows = []
for aid in THREE_IDS:
    works = artist_works[artist_works["artist_id"] == aid]
    debut  = int(works["release_date"].min())
    latest = int(works["release_date"].max())
    genres = genre_yearly_raw[genre_yearly_raw["artist_id"] == aid]["genre"].unique()
    for genre in genres:
        for yr in range(debut, latest + 1):
            genre_grid_rows.append({"artist_id": aid, "year": yr, "genre": genre})

genre_grid = pd.DataFrame(genre_grid_rows)
genre_grid["display_name"] = genre_grid["artist_id"].map(name_map)

# Merge actual counts onto full grid, fill missing years with 0
genre_yearly = genre_grid.merge(
    genre_yearly_raw[["artist_id", "year", "genre", "works_in_genre"]],
    on=["artist_id", "year", "genre"], how="left"
)
genre_yearly["works_in_genre"] = genre_yearly["works_in_genre"].fillna(0).astype(int)

# Cumulative — now every year carries forward even if no works that year
genre_yearly = genre_yearly.sort_values(["artist_id", "genre", "year"])
genre_yearly["cumulative_works_in_genre"] = (
    genre_yearly.groupby(["artist_id", "genre"])["works_in_genre"].cumsum()
)

# ── STEP 8: COMBINE INTO ONE CSV (two sections marked by section column) ───────
# Section A: timeline — one row per artist per year, all numeric metrics
timeline["section"] = "timeline"
genre_yearly["section"] = "genre"

# Align columns — fill missing cols with NaN so concat works cleanly
combined = pd.concat([timeline, genre_yearly], ignore_index=True, sort=False)

# Reorder columns so key fields are first
front_cols = [
    "section", "artist_id", "display_name", "year",
    "new_notable_works", "cumulative_notable_works",
    "new_collabs", "cumulative_collabs",
    "outbound_influence", "cumulative_outbound",
    "inbound_influence", "cumulative_inbound",
    "debut_year", "notoriety_year",
    "genre", "works_in_genre", "cumulative_works_in_genre"
]
other_cols = [c for c in combined.columns if c not in front_cols]
combined   = combined[front_cols + other_cols]

combined.to_csv("three_artists_timeline.csv", index=False)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("three_artists_timeline.csv — export summary")
print("=" * 60)
print(f"Total rows : {len(combined)}")
print(f"  Timeline section (section='timeline'): {(combined['section']=='timeline').sum()}")
print(f"  Genre section    (section='genre')   : {(combined['section']=='genre').sum()}")
print()
print("Per-artist row counts (timeline section):")
tl = combined[combined["section"] == "timeline"]
for aid, name in THREE.items():
    n = (tl["artist_id"] == aid).sum()
    yr_min = tl[tl["artist_id"]==aid]["year"].min()
    yr_max = tl[tl["artist_id"]==aid]["year"].max()
    print(f"  {name:<20} {n} rows  ({yr_min}–{yr_max})")

print()
print("Per-artist genre counts:")
gn = combined[combined["section"] == "genre"]
for aid, name in THREE.items():
    genres = gn[gn["artist_id"]==aid]["genre"].unique()
    print(f"  {name:<20} {len(genres)} genres: {list(genres)}")

print()
print("Columns in timeline section:")
print([c for c in front_cols if c not in ["genre","works_in_genre","cumulative_works_in_genre"]])
print()
print("Columns in genre section:")
print(["artist_id","display_name","year","genre","works_in_genre","cumulative_works_in_genre"])
print()
print("Done. File saved: three_artists_timeline.csv")

three_artists_timeline.csv — export summary
Total rows : 187
  Timeline section (section='timeline'): 36
  Genre section    (section='genre')   : 151

Per-artist row counts (timeline section):
  Sailor Shift         13 rows  (2028–2040)
  Min He               9 rows  (2020–2028)
  Kimberly Snyder      14 rows  (2016–2029)

Per-artist genre counts:
  Sailor Shift         1 genres: ['Oceanus Folk']
  Min He               6 genres: ['Americana', 'Desert Rock', 'Indie Folk', 'Jazz Surf Rock', 'Space Rock', 'Synthwave']
  Kimberly Snyder      6 genres: ['Dream Pop', 'Indie Rock', 'Post-Apocalyptic Folk', 'Space Rock', 'Symphonic Metal', 'Synthwave']

Columns in timeline section:
['section', 'artist_id', 'display_name', 'year', 'new_notable_works', 'cumulative_notable_works', 'new_collabs', 'cumulative_collabs', 'outbound_influence', 'cumulative_outbound', 'inbound_influence', 'cumulative_inbound', 'debut_year', 'notoriety_year']

Columns in genre section:
['artist_id', 'display_name', 'year